# Conteo de Colonias de Actinomicetos

## El problema

Contar colonias bacterianas en placas de Petri es una tarea de laboratorio muy comun. Sirve para saber cuantos microorganismos hay en una muestra. Hacerlo a mano tarda tiempo y los resultados pueden variar dependiendo de quien cuente. Por eso buscamos formas de automatizarlo.

En este proyecto trabajamos con imagenes de **actinomicetos**, un grupo de bacterias que forman colonias visibles en agar a simple vista. Cada imagen contiene **dos placas con la misma muestra**. Contar ambas y comparar los resultados sirve como indicador de confiabilidad: si los dos conteos son parecidos, el resultado es valido.

## Los datos

Tenemos 8 imagenes, cada una con dos placas (A y B). Los conteos manuales hechos en el laboratorio son los siguientes:

| Imagen | Placa A | Placa B |
|--------|---------|--------|
| actinomicetos_1 | 63 | 64 |
| actinomicetos_2 | 62 | 64 |
| actinomicetos_3 | 57 | 60 |
| actinomicetos_4 | 42 | 68 |
| actinomicetos_5 | 30 | 25 |
| actinomicetos_6 | 49 | 62 |
| actinomicetos_7 | 68 | 67 |
| actinomicetos_8 | 24 | 13 |

## Lo que se ha hecho hasta ahora

El proyecto parte de **CellSAM**, un modelo de segmentacion desarrollado por el Van Valen Lab (Caltech). El modelo combina dos partes:

- **AnchorDETR** detecta los objetos en la imagen y genera una caja alrededor de cada uno
- **SAM** (Segment Anything Model de Meta) toma cada caja y genera una mascara precisa pixel a pixel

CellSAM fue entrenado con mas de 8800 imagenes de microscopia. En este proyecto lo estamos adaptando para imagenes de placas de Petri, que son imagenes macroscopicas (visibles a simple vista), no microscopicas. Esto es un uso fuera del dominio original del modelo.

Para entender si CellSAM aporta algo en este contexto, lo comparamos con un metodo de **vision clasica**: un pipeline que no usa ninguna red neuronal, solo operaciones matematicas sobre los pixeles. Si el metodo clasico da resultados similares, quiza no necesitamos la complejidad de un modelo de deep learning. Si CellSAM es mejor, entonces vale la pena seguir explorando su uso aqui.

### El pipeline completo tiene tres etapas:

1. **Deteccion de placas:** encontrar donde estan los dos circulos (placas) en la imagen
2. **Segmentacion de colonias:** identificar cada colonia dentro de cada placa
3. **Filtrado:** descartar regiones que no son colonias (demasiado pequenas, demasiado grandes, o con forma irregular)

## 1. Importaciones

In [ ]:
import sys
import csv
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops, label as sk_label
from cellSAM import get_model, segment_cellular_image

# Rutas del proyecto
IMAGES_DIR  = Path('images/placas')
GT_CSV      = IMAGES_DIR / 'ground_truth.csv'
OUTPUT_DIR  = Path('results/colonies')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Librerias cargadas.')

## 2. Parametros

Estos valores controlan el comportamiento de ambos metodos. Si los conteos son consistentemente muy altos o muy bajos, estos son los primeros valores a ajustar.

In [ ]:
# Filtrado (compartido por ambos metodos)
MIN_COLONY_AREA    = 300    # px2: regiones mas pequenas que esto se descartan como ruido
MAX_COLONY_AREA    = 50000  # px2: regiones mas grandes son colonias pegadas o artefactos
MIN_SOLIDITY       = 0.50   # 0 a 1: filtra formas muy irregulares (bordes del vidrio)

# Metodo clasico
TOPHAT_KERNEL      = 81     # px: debe ser mas grande que la colonia mas grande
TOPHAT_THRESHOLD   = 10     # umbral sobre la salida del top-hat
SAT_THRESHOLD      = 60     # umbral de saturacion HSV para colonias de color
MAX_HOLE_AREA      = 400    # px2: huecos mas pequenos que esto se rellenan
OPEN_ITERATIONS    = 1      # pasadas de apertura morfologica para quitar ruido
WATERSHED_MIN_DIST = 20     # px: distancia minima entre centros de colonias

# CellSAM
NORMALIZE          = True   # normalizacion por percentil + CLAHE
POSTPROCESS        = False  # postprocesamiento extra (recomendado para imagenes ruidosas)

print('Parametros definidos.')

## 3. Cargar el modelo CellSAM

`get_model()` descarga los pesos del modelo si no estan en cache, o los carga desde `~/.deepcell/models/` si ya se descargaron antes. El modelo se carga **una sola vez** y se reutiliza para todas las imagenes y placas.

In [ ]:
print('Cargando modelo CellSAM...')
model = get_model()
print('Modelo listo.')

## 4. Deteccion de placas

Cada imagen tiene dos placas circulares. Para encontrarlas usamos **HoughCircles**, un algoritmo clasico que detecta circulos en una imagen en escala de grises.

Para evitar que detecte dos circulos en la misma placa, dividimos la imagen en dos mitades antes de buscar. Si la imagen es vertical (mas alta que ancha) buscamos en la mitad superior e inferior. Si es horizontal buscamos en la mitad izquierda y derecha.

Una vez encontrado el circulo, recortamos esa region y ponemos a negro todo lo que esta fuera del circulo, para que el segmentador solo vea el contenido de la placa.

In [ ]:
def detect_plates(img_bgr):
    h, w     = img_bgr.shape[:2]
    portrait = h > w
    plates   = []
    for i in range(2):
        if portrait:
            y0, y1 = i * h // 2, (i + 1) * h // 2
            half   = img_bgr[y0:y1, :]
            ox, oy = 0, y0
        else:
            x0, x1 = i * w // 2, (i + 1) * w // 2
            half   = img_bgr[:, x0:x1]
            ox, oy = x0, 0
        hh, hw  = half.shape[:2]
        gray    = cv2.cvtColor(half, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (21, 21), 0)
        circles = cv2.HoughCircles(
            blurred, cv2.HOUGH_GRADIENT, dp=1.2,
            minDist=max(hh, hw), param1=60, param2=25,
            minRadius=int(min(hh, hw) * 0.30),
            maxRadius=int(min(hh, hw) * 0.52),
        )
        if circles is not None:
            best = np.round(circles[0][0]).astype(int)
            cx, cy, r = int(best[0]) + ox, int(best[1]) + oy, int(best[2])
        else:
            print(f'  Aviso: no se encontro circulo en mitad {i+1}, usando centro.')
            cx, cy, r = hw // 2 + ox, hh // 2 + oy, int(min(hh, hw) * 0.43)
        plates.append((cx, cy, r))
    return plates


def crop_plate(img_bgr, cx, cy, r, shrink=0.86):
    r_use = int(r * shrink)
    x1 = max(0, cx - r_use);  y1 = max(0, cy - r_use)
    x2 = min(img_bgr.shape[1], cx + r_use)
    y2 = min(img_bgr.shape[0], cy + r_use)
    crop   = img_bgr[y1:y2, x1:x2].copy()
    hc, wc = crop.shape[:2]
    mask   = np.zeros((hc, wc), dtype=np.uint8)
    cv2.circle(mask, (cx - x1, cy - y1), r_use, 255, -1)
    crop[mask == 0] = 0
    return crop, mask


print('Funciones de deteccion definidas.')

## 5. Metodo 1: Vision clasica

Este metodo no usa ninguna red neuronal. Detecta colonias a partir de sus propiedades visuales usando operaciones matematicas paso a paso:

1. **Gaussian blur 5x5:** suaviza el ruido de la imagen antes de procesar
2. **Top-hat morfologico en escala de grises (kernel 81px):** extrae las manchas que son mas brillantes que su entorno local. Esto funciona bien para colonias blancas o crema sobre agar mas oscuro, sin importar si hay variacion de iluminacion en la placa
3. **Umbral fijo sobre el top-hat:** convierte el resultado en una imagen binaria (blanco/negro). Todo lo que supera el umbral es posible colonia
4. **Umbral de saturacion HSV:** detecta colonias de color (rosa, naranja, amarillo) que no necesariamente son mas brillantes que el fondo pero si mas saturadas. Se combina con el resultado anterior usando OR
5. **Relleno de huecos pequenos (menos de 400 px2):** muchas colonias tienen un punto oscuro en el centro. Sin este paso, cada colonia aparece como un anillo en lugar de un disco lleno
6. **Apertura morfologica:** elimina pequenos puntos de ruido que pasaron el umbral
7. **Watershed:** separa colonias que se tocan entre si, usando la distancia al borde de cada colonia para encontrar los centros y cortar ahi

In [ ]:
def segment_classical(crop_bgr, plate_mask):
    # Paso 1+2: blur y top-hat en escala de grises
    gray    = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray    = cv2.bitwise_and(gray, gray, mask=plate_mask)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (TOPHAT_KERNEL, TOPHAT_KERNEL))
    tophat  = cv2.morphologyEx(blurred, cv2.MORPH_TOPHAT, kernel)

    # Paso 3: umbral sobre top-hat
    _, bin_lum = cv2.threshold(tophat, TOPHAT_THRESHOLD, 255, cv2.THRESH_BINARY)
    bin_lum    = cv2.bitwise_and(bin_lum, bin_lum, mask=plate_mask)

    # Paso 4: saturacion HSV para colonias de color
    hsv        = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    sat        = cv2.bitwise_and(hsv[:, :, 1], hsv[:, :, 1], mask=plate_mask)
    _, bin_col = cv2.threshold(sat, SAT_THRESHOLD, 255, cv2.THRESH_BINARY)

    # Combinacion OR
    binary = cv2.bitwise_or(bin_lum, bin_col)
    binary = cv2.bitwise_and(binary, binary, mask=plate_mask)

    # Paso 5: relleno de huecos pequenos
    inv_labeled   = sk_label(~binary.astype(bool))
    binary_filled = binary.copy()
    for rid in range(1, inv_labeled.max() + 1):
        hole = inv_labeled == rid
        if hole.sum() <= MAX_HOLE_AREA:
            binary_filled[hole] = 255

    # Paso 6: apertura morfologica
    k3            = np.ones((3, 3), np.uint8)
    binary_filled = cv2.morphologyEx(binary_filled, cv2.MORPH_OPEN, k3, iterations=OPEN_ITERATIONS)
    binary_filled = cv2.bitwise_and(binary_filled, binary_filled, mask=plate_mask)

    # Paso 7: watershed
    dist   = ndimage.distance_transform_edt(binary_filled)
    coords = peak_local_max(dist, min_distance=WATERSHED_MIN_DIST,
                            threshold_rel=0.15, labels=binary_filled.astype(bool))
    if len(coords) == 0:
        _, labels_out = cv2.connectedComponents(binary_filled)
        props = regionprops(labels_out)
        valid = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
                 and p.solidity >= MIN_SOLIDITY]
        return labels_out, valid, binary_filled

    local_max              = np.zeros_like(dist, dtype=bool)
    local_max[tuple(coords.T)] = True
    markers                = sk_label(local_max)
    labels_ws              = watershed(-dist, markers, mask=binary_filled.astype(bool))
    props = regionprops(labels_ws)
    valid = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
             and p.solidity >= MIN_SOLIDITY]
    return labels_ws, valid, binary_filled


print('Funcion de segmentacion clasica definida.')

## 6. Metodo 2: CellSAM

En lugar de operar pixel a pixel, CellSAM entiende el contenido de la imagen como un todo. Recibe el recorte de la placa en RGB, genera cajas alrededor de cada objeto que detecta, y luego produce una mascara precisa para cada uno.

El parametro `normalize=True` aplica una normalizacion por percentil seguida de CLAHE (ecualizado adaptativo del histograma) antes de pasar la imagen al modelo. Esto puede mejorar el contraste en imagenes con iluminacion despareja.

Despues de que CellSAM devuelve su mascara, aplicamos el mismo filtro de area y solidez que en el metodo clasico para descartar detecciones que claramente no son colonias.

In [ ]:
def segment_cellsam(crop_bgr, plate_mask):
    crop_rgb   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    mask, _, _ = segment_cellular_image(
        crop_rgb, model=model,
        normalize=NORMALIZE, postprocess=POSTPROCESS, device='cpu',
    )
    mask = mask.copy()
    mask[plate_mask == 0] = 0
    binary = (mask > 0).astype(np.uint8) * 255
    props  = regionprops(mask)
    valid  = [p for p in props if MIN_COLONY_AREA <= p.area <= MAX_COLONY_AREA
              and p.solidity >= MIN_SOLIDITY]
    return mask, valid, binary


def draw_overlay(crop_bgr, valid_props, label_mask):
    overlay = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    for prop in valid_props:
        region = label_mask == prop.label
        overlay[region] = overlay[region] * 0.45 + np.array([0.15, 0.85, 0.35]) * 0.55
    return overlay


print('Funciones de CellSAM definidas.')

## 7. Cargar los conteos manuales (ground truth)

El ground truth son los conteos hechos a mano en el laboratorio. Los usamos como referencia para evaluar que tan cerca llega cada metodo al valor real.

In [ ]:
gt = {}
with open(GT_CSV, newline='', encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        stem = Path(row['image']).stem
        gt[stem] = (int(row['plate_A']), int(row['plate_B']))

print(f'Ground truth cargado: {len(gt)} imagenes')
for k, v in gt.items():
    print(f'  {k}: A={v[0]}, B={v[1]}')

## 8. Procesar todas las imagenes

Para cada imagen corremos los dos metodos en ambas placas y guardamos los resultados. El proceso por imagen es:

1. Detectar los dos circulos
2. Recortar cada placa
3. Segmentar con metodo clasico y con CellSAM
4. Guardar conteos y generar una imagen comparativa

In [ ]:
SUPPORTED = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
images    = sorted(p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in SUPPORTED)
print(f'Imagenes encontradas: {len(images)}\n')

results = []

for idx, img_path in enumerate(images, 1):
    print(f'[{idx}/{len(images)}] {img_path.name}')
    img    = cv2.imread(str(img_path))
    plates = detect_plates(img)

    counts_cl, counts_cs = [], []
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(img_path.name, fontsize=14, fontweight='bold')

    for i, (cx, cy, r) in enumerate(plates):
        name       = ['A', 'B'][i]
        crop, mask = crop_plate(img, cx, cy, r)

        lbl_cl, val_cl, _ = segment_classical(crop, mask)
        lbl_cs, val_cs, _ = segment_cellsam(crop, mask)

        n_cl, n_cs = len(val_cl), len(val_cs)
        counts_cl.append(n_cl)
        counts_cs.append(n_cs)
        print(f'  Placa {name}: clasico={n_cl}  CellSAM={n_cs}')

        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(crop_rgb)
        axes[i, 0].set_title(f'Placa {name} - original', fontsize=11)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(draw_overlay(crop, val_cl, lbl_cl))
        axes[i, 1].set_title(f'Clasico: {n_cl} colonias', fontsize=11)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(draw_overlay(crop, val_cs, lbl_cs))
        axes[i, 2].set_title(f'CellSAM: {n_cs} colonias', fontsize=11)
        axes[i, 2].axis('off')

    plt.tight_layout()
    out = OUTPUT_DIR / f'{img_path.stem}_comparison.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()

    row = {
        'name':      img_path.stem,
        'classical': counts_cl,
        'cellsam':   counts_cs,
    }
    if img_path.stem in gt:
        row['gt'] = gt[img_path.stem]
    results.append(row)
    print()

print('Procesamiento completado.')

## 9. Tabla de resultados

In [ ]:
rows = []
for r in results:
    ga, gb = r['gt'] if r.get('gt') else (None, None)
    ca, cb = r['classical']
    sa, sb = r['cellsam']
    rows.append({
        'Imagen':       r['name'],
        'GT A':         ga,
        'GT B':         gb,
        'Clasico A':    ca,
        'Clasico B':    cb,
        'CellSAM A':    sa,
        'CellSAM B':    sb,
        'Err Clasico A': (ca - ga) if ga is not None else None,
        'Err Clasico B': (cb - gb) if gb is not None else None,
        'Err CellSAM A': (sa - ga) if ga is not None else None,
        'Err CellSAM B': (sb - gb) if gb is not None else None,
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / 'comparison_summary.csv', index=False)
df

## 10. Grafico comparativo

Tres barras por imagen: verde = conteo manual, azul = metodo clasico, naranja = CellSAM. La barra verde es el valor de referencia al que queremos acercarnos.

In [ ]:
images_names = [r['name'].replace('actinomicetos_', 'actin_') for r in results]
x     = np.arange(len(results))
width = 0.22

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)
fig.suptitle('Comparativa de metodos de conteo de colonias', fontsize=14, fontweight='bold')

for ax, plate_idx, plate_label in [(ax1, 0, 'Placa A'), (ax2, 1, 'Placa B')]:
    gt_vals  = [r['gt'][plate_idx]        if r.get('gt') else 0 for r in results]
    cl_vals  = [r['classical'][plate_idx]                        for r in results]
    cs_vals  = [r['cellsam'][plate_idx]                          for r in results]
    all_vals = gt_vals + cl_vals + cs_vals

    b0 = ax.bar(x - width,     gt_vals, width, label='Ground truth', color='#2ca02c', alpha=0.85)
    b1 = ax.bar(x,             cl_vals, width, label='Clasico CV',   color='#4C72B0', alpha=0.85)
    b2 = ax.bar(x + width,     cs_vals, width, label='CellSAM',      color='#DD8452', alpha=0.85)

    for bars in [b0, b1, b2]:
        ax.bar_label(bars, padding=3, fontsize=8)

    ax.set_ylabel('Colonias contadas')
    ax.set_title(plate_label, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(all_vals, default=1) * 1.25)
    ax.grid(axis='y', alpha=0.3)

ax2.set_xticks(x)
ax2.set_xticklabels(images_names, rotation=30, ha='right', fontsize=9)
plt.tight_layout()
out = OUTPUT_DIR / 'comparison_chart.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico guardado: {out}')

## 11. Error promedio por metodo

In [ ]:
errs_cl, errs_cs = [], []
for r in results:
    if not r.get('gt'):
        continue
    for i in range(2):
        errs_cl.append(abs(r['classical'][i] - r['gt'][i]))
        errs_cs.append(abs(r['cellsam'][i]   - r['gt'][i]))

print('=' * 45)
print('  ERROR ABSOLUTO MEDIO vs GROUND TRUTH')
print('=' * 45)
print(f'  Metodo clasico: {np.mean(errs_cl):.1f} colonias por placa')
print(f'  CellSAM:        {np.mean(errs_cs):.1f} colonias por placa')
print('=' * 45)

## 12. Que encontramos

Ambos metodos intentan detectar colonias en imagenes para las que no fueron disenados originalmente:

- El **metodo clasico** fue ajustado manualmente con parametros que funcionan para colonias blancas o crema. Es rapido y no requiere GPU, pero depende de que las condiciones de imagen sean parecidas entre fotos.

- **CellSAM** fue entrenado con imagenes de microscopia, no con placas de Petri macroscopicas. Aun asi puede detectar objetos circulares y con bordes definidos, que es lo que son las colonias. Su ventaja es que no necesita parametros ajustados a mano.

La comparativa con el ground truth nos muestra que tan lejos esta cada metodo del conteo real, lo que nos da informacion para decidir que camino seguir.

## 13. Pasos a futuro

### Opcion 1: Ajustar los parametros del metodo clasico
Si el metodo clasico queda cerca del ground truth, se pueden ajustar los parametros (`TOPHAT_THRESHOLD`, `WATERSHED_MIN_DIST`, `MIN_COLONY_AREA`) para mejorar los casos donde falla. Es la opcion mas rapida.

### Opcion 2: Ajustar los parametros de CellSAM
CellSAM tiene parametros que no hemos explorado todavia:
- `postprocess=True`: activa un postprocesamiento extra para imagenes ruidosas
- `bbox_threshold`: controla que tan seguros deben ser los bounding boxes (default 0.4). Bajarlo detecta mas objetos
- `normalize=False`: desactiva el preprocesamiento interno, util si las imagenes ya tienen buen contraste

### Opcion 3: Fine-tuning de CellSAM
Si ninguno de los dos metodos llega al nivel deseado, se puede hacer fine-tuning de CellSAM con las imagenes de actinomicetos etiquetadas. Esto requiere crear mascaras de segmentacion para las colonias (una por colonia, no solo el conteo) y entrenar el modelo unas pocas iteraciones. Con pocas imagenes bien anotadas puede mejorar significativamente el rendimiento en este tipo especifico de placa.

### Opcion 4: Expandir a otras bacterias
Una vez que el pipeline funciona bien para actinomicetos, se puede adaptar para otras bacterias (E. coli, Bacillus, etc.) ajustando los parametros de tamano y forma de las colonias.